# قاعدة بيانات نواة 🌴
هذا الدفتر يفتح قاعدة البيانات `nawah.db` (قاعدة SQL من نوع SQLite) ويعرض جداولها ويسمح لك بتعديلها.

**طريقة الفتح من Anaconda:**
1. افتحي **Anaconda Navigator** ثم **Jupyter Notebook** (أو **JupyterLab**).
2. ادخلي إلى مجلد المشروع `nawah` وافتحي هذا الملف `view_database.ipynb`.
3. شغّلي الخلايا من الأعلى إلى الأسفل (زر ▶ أو `Shift + Enter`).

> الدفتر يجب أن يبقى داخل مجلد `nawah` بجانب ملف `nawah.db`.

In [ ]:
import sqlite3
from pathlib import Path

DB_FILE = "nawah.db"
if not Path(DB_FILE).exists():
    raise FileNotFoundError("ما لقيت nawah.db. افتحي هذا الدفتر من داخل مجلد nawah.")

conn = sqlite3.connect(DB_FILE)
print("تم الاتصال بقاعدة البيانات:", Path(DB_FILE).resolve())

In [ ]:
# دالتان مساعدتان: q لعرض البيانات (SELECT)، و run لتعديلها (INSERT / UPDATE / DELETE)
try:
    import pandas as pd
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)

    def q(sql, params=()):
        """تشغّل أمر SELECT وترجع النتيجة كجدول."""
        return pd.read_sql_query(sql, conn, params=params)
except Exception:
    def q(sql, params=()):
        """نسخة بديلة بدون pandas: تطبع الصفوف كما هي."""
        cur = conn.execute(sql, params)
        print([d[0] for d in cur.description])
        for row in cur.fetchall():
            print(row)

def run(sql, params=()):
    """تشغّل أمر تعديل وتحفظه في القاعدة."""
    cur = conn.execute(sql, params)
    conn.commit()
    print("تم. عدد الصفوف المتأثرة:", cur.rowcount)

## 1) الجداول الموجودة

In [ ]:
q("SELECT name AS table_name FROM sqlite_master WHERE type = 'table' ORDER BY name")

## 2) أنواع التمور وقيمها الغذائية (`varieties`)

In [ ]:
q("SELECT * FROM varieties")

## 3) الشركات (`companies`)
الأعمدة `rating` (التقييم من 10) و`featured` (1 = مميزة) تتحكم في ظهور الشركة في صفحة الشركات.

In [ ]:
q("SELECT id, name, name_en, country, city, founded_year, varieties, rating, featured FROM companies ORDER BY featured DESC, rating DESC")

## 4) اقتراحات الشركات (`suggestions`)

In [ ]:
q("SELECT * FROM suggestions ORDER BY id DESC")

## 5) سجل تحليل الصور (`analyses`)
`user_confirmed`: 1 = المستخدم أكد أن النوع صحيح، 0 = خطأ، فارغ = لم يجب بعد.

In [ ]:
q("""
SELECT a.id, a.created_at, a.ai_variety, a.ai_confidence, a.user_confirmed, a.corrected_variety, v.name_en
FROM analyses a
LEFT JOIN varieties v ON v.id = a.variety_id
ORDER BY a.id DESC
""")

## 6) استعلامات مفيدة

In [ ]:
q("SELECT country, COUNT(*) AS companies_count FROM companies GROUP BY country ORDER BY companies_count DESC")

In [ ]:
q("SELECT COUNT(*) AS total, SUM(user_confirmed IS NOT NULL) AS answered, SUM(user_confirmed = 1) AS correct FROM analyses")

## 7) تعديل البيانات
أزيلي علامة `#` من السطر الذي تحتاجينه ثم شغّلي الخلية. **التعديل يظهر في الموقع مباشرة** بعد تحديث الصفحة.

In [ ]:
# تغيير تقييم شركة:
# run("UPDATE companies SET rating = ? WHERE id = ?", (9.5, 1))

# جعل شركة مميزة (1) أو عادية (0):
# run("UPDATE companies SET featured = ? WHERE id = ?", (1, 3))

# إضافة شركة جديدة (الأنواع تُكتب بفاصلة عربية: مجدول،برحي):
# run("INSERT INTO companies (name, name_en, country, city, varieties, rating, featured) VALUES (?, ?, ?, ?, ?, ?, ?)",
#     ("اسم الشركة", "Company name", "السعودية", "الرياض", "سكري،خلاص", 8.0, 0))

# حذف شركة:
# run("DELETE FROM companies WHERE id = ?", (5,))

## 8) تصدير إلى Excel
ينشئ ملفاً بجانب الدفتر يفتح في Excel بالعربي بشكل سليم.

In [ ]:
q("SELECT * FROM companies").to_csv("companies.csv", index=False, encoding="utf-8-sig")
q("SELECT * FROM analyses").to_csv("analyses.csv", index=False, encoding="utf-8-sig")
print("تم إنشاء companies.csv و analyses.csv")